In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Exercise02") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/31 23:20:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
user_df = spark.read.csv("data/Users.csv", header=True, inferSchema=True, sep=";")
user_df.printSchema()

root
 |-- UserID: integer (nullable = true)
 |-- USERNAME: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Age: string (nullable = true)



In [9]:
user_df.show(5)

+------+-----------+--------------------+----+
|UserID|   USERNAME|            Location| Age|
+------+-----------+--------------------+----+
|     1|bzsufoRTLN2|  nyc, new york, usa|NULL|
|     2|fq7kfHg4VEI|stockton, califor...|  18|
|     3|W0Hbkd3xR8v|moscow, yukon ter...|NULL|
|     4|W51GahAx5Ap|porto, v.n.gaia, ...|  17|
|     5|VKN3PQ18GgN|farnborough, hant...|NULL|
+------+-----------+--------------------+----+
only showing top 5 rows


In [10]:
book_df = spark.read.csv("data/Books.csv", header=True, inferSchema=True, sep=";")
book_df.printSchema()
book_df.show(5)

root
 |-- ISBN: string (nullable = true)
 |-- BookTitle: string (nullable = true)
 |-- BookAuthor: string (nullable = true)
 |-- YearOfPublication: integer (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- ImageURLS: string (nullable = true)
 |-- ImageURLM: string (nullable = true)
 |-- ImageURLL: string (nullable = true)

+----------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|      ISBN|           BookTitle|          BookAuthor|YearOfPublication|           Publisher|           ImageURLS|           ImageURLM|           ImageURLL|
+----------+--------------------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+
|0195153448| Classical Mythology|  Mark P. O. Morford|             2002|Oxford University...|http://images.ama...|http://images.ama...|http://images.ama...|
|0002005018|        Clara Callan|

In [11]:
rating_df = spark.read.csv("data/Book-Ratings.csv", header=True, inferSchema=True, sep=";")
rating_df.printSchema()
rating_df.show(5)

root
 |-- userid: integer (nullable = true)
 |-- isbn: string (nullable = true)
 |-- rate: integer (nullable = true)

+------+----------+----+
|userid|      isbn|rate|
+------+----------+----+
|276725|034545104X|   0|
|276726|0155061224|   5|
|276727|0446520802|   0|
|276729|052165615X|   3|
|276729|0521795028|   6|
+------+----------+----+
only showing top 5 rows


In [14]:
from pyspark.sql import Window
from pyspark.sql.functions import avg

book_window = Window.partitionBy(book_df.ISBN)

named_rating_df = rating_df \
    .join(user_df, rating_df.userid == user_df.UserID) \
    .join(book_df, rating_df.isbn == book_df.ISBN) \
    .withColumn("BookAvgRate", avg(rating_df.rate).over(book_window)) \
    .select(
        user_df.USERNAME,
        book_df.BookTitle,
        rating_df.rate,
        "BookAvgRate"
    )

named_rating_df.show(15, truncate=False)

+-----------+----------------------------------+----+------------------+
|USERNAME   |BookTitle                         |rate|BookAvgRate       |
+-----------+----------------------------------+----+------------------+
|cHwJip4Kj4k|The Fighting Man                  |9   |9.0               |
|cHwJip4Kj4k|Nothing Can Be Better             |0   |0.0               |
|vRhKAqzPu1g|Which Colour?                     |0   |0.0               |
|1er18zMrBmP|Huck Scarry's Steam Train Journey |0   |0.0               |
|UfA21idiBYZ|The Big Honey Hunt                |0   |0.0               |
|5wTiYF3rSw0|ARE YOU MY MOTHER MINI PB (EXPORT)|0   |0.0               |
|LAEKIXXYrb9|Bess                              |0   |0.0               |
|bwo9jnhH1hi|THE COAL HOUSE T/PB               |0   |0.0               |
|NQPC9CE4kNW|THE COAL HOUSE T/PB               |0   |0.0               |
|hOx25KyXZoR|THE COAL HOUSE T/PB               |0   |0.0               |
|bwo9jnhH1hi|Glue (First Facts - First Skills) |0  

In [15]:
rating_df.createOrReplaceTempView("rating_df")
user_df.createOrReplaceTempView("user_df")
book_df.createOrReplaceTempView("book_df")

named_rating_df = spark.sql("""
    SELECT
        u.USERNAME,
        b.BookTitle,
        r.rate,
        AVG(r.rate) OVER (PARTITION BY b.ISBN) AS BookAvgRate
    FROM rating_df r
    JOIN user_df u ON r.userid = u.UserID
    JOIN book_df b ON r.isbn = b.ISBN
""")

named_rating_df.show(15, truncate=False)

+-----------+----------------------------------+----+------------------+
|USERNAME   |BookTitle                         |rate|BookAvgRate       |
+-----------+----------------------------------+----+------------------+
|cHwJip4Kj4k|The Fighting Man                  |9   |9.0               |
|cHwJip4Kj4k|Nothing Can Be Better             |0   |0.0               |
|vRhKAqzPu1g|Which Colour?                     |0   |0.0               |
|1er18zMrBmP|Huck Scarry's Steam Train Journey |0   |0.0               |
|UfA21idiBYZ|The Big Honey Hunt                |0   |0.0               |
|5wTiYF3rSw0|ARE YOU MY MOTHER MINI PB (EXPORT)|0   |0.0               |
|LAEKIXXYrb9|Bess                              |0   |0.0               |
|bwo9jnhH1hi|THE COAL HOUSE T/PB               |0   |0.0               |
|NQPC9CE4kNW|THE COAL HOUSE T/PB               |0   |0.0               |
|hOx25KyXZoR|THE COAL HOUSE T/PB               |0   |0.0               |
|bwo9jnhH1hi|Glue (First Facts - First Skills) |0  